# Project II: Economic Growth 

In [1]:
### A. Settings
%load_ext autoreload
%autoreload 2

# i. Importing packages
import pandas as pd 
import numpy as np 
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from numpy import linalg as la
from tabulate import tabulate
from scipy import stats
from pathlib import Path

# ii. Importing methods from python script
import Project2

# iii. Plotting settings
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.family':'serif','mathtext.fontset':'cm'})
plt.rcParams.update({'font.size': 12})


## Read data 

In [2]:
dat = pd.read_csv('growth.csv')
lbldf = pd.read_csv('labels.csv', index_col='variable')
lbl_all = lbldf.label.to_dict() # as a dictionary
print(f'The data contains {dat.shape[0]} rows (countries) and {dat.shape[1]} columns (variables).')

The data contains 214 rows (countries) and 85 columns (variables).


In [3]:
# convenient to keep a column of ones in the dataset
dat['constant'] = np.ones((dat.shape[0],))

# Collections of variables

In order to make the analysis simpler, it may be convenient to collect variables in sets that belong together naturally. 

In [4]:
# all available variables
vv_institutions = ['marketref', 'dem', 'demCGV', 'demBMR', 'demreg'] 
vv_geography = [
        'tropicar','distr', 'distcr', 'distc','suitavg','temp', 'suitgini', 'elevavg', 'elevstd',
        'kgatr', 'precip', 'area', 'abslat', 'cenlong', 'area_ar', 'rough','landlock', 
        'africa',  'asia', 'oceania', 'americas' # 'europe' is the reference
]
vv_geneticdiversity = ['pdiv', 'pdiv_aa', 'pdivhmi', 'pdivhmi_aa']
vv_historical = ['pd1000', 'pd1500', 'pop1000', 'pop1500', 'ln_yst'] # these are often missing: ['pd1', 'pop1']
vv_religion = ['pprotest', 'pcatholic', 'pmuslim']
vv_danger = ['yellow', 'malfal',  'uvdamage']
vv_resources = ['oilres', 'goldm', 'iron', 'silv', 'zinc']
vv_educ = ['ls_bl', 'lh_bl'] # secondary, tertiary: we exclude 'lp_bl' (primary) to avoid rank failure 

vv_all = {'institutions': vv_institutions, 
          'geography': vv_geography, 
          'geneticdiversity': vv_geneticdiversity,
          'historical': vv_historical,
          'religion': vv_religion,
          'danger':vv_danger, 
          'resources':vv_resources
         }
list_of_lists = vv_all.values()
vv_all['all'] = [v for sublist in list_of_lists for v in sublist]


# Selection of controls

This section reduces the candidate controls before the post-double-selection lasso. No convergence regression is estimated here. Each step adds the variables it drops to `vv_dropped`; `dat` itself is not changed.

## 1. Load and inspect

`dat` and `lbl_all` are used as loaded above. The header of `growth.csv` lists `pd1500` twice, and pandas names the second copy `pd1500.1`. If the copy equals `pd1500`, it carries no extra information and is dropped.

In [5]:
# 1. pandas reads the second pd1500 column as pd1500.1
print(f"pd1500 and pd1500.1 are equal: {dat['pd1500'].equals(dat['pd1500.1'])}")

# 2. drop the copy
vv_dropped = {'pd1500.1': 'load'}

pd1500 and pd1500.1 are equal: True


## 2. Trim the sample

The convergence regression needs both `gdp_growth` and `lgdp_initial`, so all later steps use only the countries where both are observed (`dat_trim`).

In [6]:
# 1. countries with both outcome and regressor
I = dat[['gdp_growth', 'lgdp_initial']].notnull().all(axis=1)

# 2. trimmed sample without the dropped copy
dat_trim = dat.loc[I, [v for v in dat.columns if v not in vv_dropped]]
print(f'Countries before trimming: {dat.shape[0]}, after trimming: {dat_trim.shape[0]}')

Countries before trimming: 214, after trimming: 102


## 3. Solow core variables

One variable per Solow concept is protected: it is never dropped in the steps below. `lgdp_initial` is protected as the regressor of interest.

### Economic theory

[To be written by the group]

In [7]:
# 1. one variable per Solow concept
vv_solow = {'Population growth': 'pop_growth',
            'Saving / interest rate': 'investment_rate',
            'Technology growth (proxy)': 'lh_bl',
            'Human capital': 'ls_bl'}

# 2. protected variables are never dropped
protected = ['lgdp_initial'] + list(vv_solow.values())

# 3. non-missing values in the trimmed sample
pd.DataFrame({'concept': list(vv_solow.keys()),
              'chosen': list(vv_solow.values()),
              'non-missing': dat_trim[list(vv_solow.values())].notnull().sum().values})

,concept,chosen,non-missing
0,Population growth,pop_growth,102
1,Saving / interest rate,investment_rate,99
2,Technology growth (proxy),lh_bl,88
3,Human capital,ls_bl,88


| Concept | Candidates in the data | Why the chosen variable |
|---|---|---|
| Population growth | `pop_growth`, `population_initial`, `population_now` | `pop_growth` is the growth rate itself; the two levels are only its start and end points. |
| Saving / interest rate | `investment_rate`, `ginv`, `capital_growth_pct_gdp_initial`, `capital_growth_pct_gdp_now` (no interest rate in the data) | `investment_rate` is the investment share of GDP averaged over the whole growth period, the measure named in the assignment. The other investment measures are not protected and go through the steps below like any other control. |
| Technology growth | `lh_bl`; institutions: `polity`, `polity2`, `dem`, `demBMR`, `demCGV`, `marketref`, `currentinst`, `legor_uk`, `legor_fr`; history: `ln_yst`, `pd1500` | No direct measure exists. `lh_bl` (proxy): a workforce with tertiary education is what develops and adopts new technology. |
| Human capital | `ls_bl`, `lp_bl`, `lh_bl`, `leb95`, `imr95` | `ls_bl`: secondary education is the broadest skill measure not already used as the technology proxy; `leb95` and `imr95` are health measures from 1995. |

Capital is not a separate Solow variable: in the Solow model capital accumulates through saving, so the saving rate already covers it.


## 4. Mechanical identities and leakage

Variables linked by definition carry the same information. The data come from different sources, so the relations are applied by definition and are not tested on the data. Rule: drop the non-protected variable with the most missing values in the trimmed sample. On a tie, drop the derived variable or, for a set that sums to a constant, the base category already used in the notebook. Variables that construct the outcome or the regressor are always dropped (leakage). Numbers in brackets are missing values in the trimmed sample.

In [8]:
# 1. relations by definition: relation, linked variables, dropped variables
vv_rel = [
    ['leakage: construct the regressor', ['gdp_pc_initial', 'gdp_initial'], ['gdp_pc_initial', 'gdp_initial']],
    ['leakage: construct the outcome', ['gdp_pc_now', 'gdp_now'], ['gdp_pc_now', 'gdp_now']],
    ['lpop_initial = ln(population_initial)', ['lpop_initial', 'population_initial'], ['lpop_initial']],
    ['pop_growth from population_initial and population_now',
     ['pop_growth', 'population_initial', 'population_now'], ['population_now']],
    ['mortality, its log and exp(log)', ['mortality', 'logem4', 'imputedmort'], ['mortality', 'imputedmort']],
    ['pd1 = pop1 / area', ['pd1', 'pop1', 'area'], ['pd1']],
    ['pd1000 = pop1000 / area', ['pd1000', 'pop1000', 'area'], ['pd1000']],
    ['pd1500 = pop1500 / area', ['pd1500', 'pop1500', 'area'], ['pd1500']],
    ['continent dummies sum to 1', ['africa', 'asia', 'oceania', 'americas', 'europe'], ['europe']],
    ['religion shares sum to 100', ['pprotest', 'pcatholic', 'pmuslim', 'pother'], ['pother']],
    ['Barro-Lee education shares', ['lp_bl', 'ls_bl', 'lh_bl'], ['lp_bl']],
    ['polity2 = polity recoded', ['polity', 'polity2'], ['polity']],
    ['distcr from distc and distr', ['distcr', 'distc', 'distr'], ['distcr']],
]

# 2. add the dropped variables to vv_dropped
for rel, vv, drop in vv_rel:
    for v in drop:
        vv_dropped[v] = 'leakage' if rel.startswith('leakage') else 'identity'


# 3. assert that non-missing polity2 values are within the -10 to 10 scale
assert dat_trim['polity2'].dropna().between(-10, 10).all(), \
    "polity2 contains non-missing values outside the -10 to 10 scale"

# 4. table with missing values in the trimmed sample in brackets
n_miss = dat_trim.isnull().sum()
pd.DataFrame({
    'relation': [r[0] for r in vv_rel],
    'variables': [', '.join(f'{v} ({n_miss[v]})' for v in r[1]) for r in vv_rel],
    'dropped': [', '.join(r[2]) for r in vv_rel]
})

,relation,variables,dropped
0,leakage: construct the regressor,"gdp_pc_initial (0), gdp_initial (0)","gdp_pc_initial, gdp_initial"
1,leakage: construct the outcome,"gdp_pc_now (0), gdp_now (0)","gdp_pc_now, gdp_now"
2,lpop_initial = ln(population_initial),"lpop_initial (0), population_initial (0)",lpop_initial
3,pop_growth from population_initial and populat...,"pop_growth (0), population_initial (0), popula...",population_now
4,"mortality, its log and exp(log)","mortality (51), logem4 (31), imputedmort (35)","mortality, imputedmort"
5,pd1 = pop1 / area,"pd1 (17), pop1 (17), area (0)",pd1
6,pd1000 = pop1000 / area,"pd1000 (7), pop1000 (7), area (0)",pd1000
7,pd1500 = pop1500 / area,"pd1500 (3), pop1500 (3), area (0)",pd1500
8,continent dummies sum to 1,"africa (0), asia (0), oceania (0), americas (0...",europe
9,religion shares sum to 100,"pprotest (0), pcatholic (0), pmuslim (0), poth...",pother


- **Leakage (regressor):** `gdp_pc_initial` is `lgdp_initial` before taking logs, and `gdp_initial` is `gdp_pc_initial` times population. Both are dropped.
- **Leakage (outcome):** `gdp_pc_now` and `gdp_pc_initial` together give `gdp_growth`, and `gdp_now` is `gdp_pc_now` times population. Both are dropped.
- **`lpop_initial` = ln(`population_initial`):** neither is missing, so the derived `lpop_initial` is dropped and population enters in levels.
- **`pop_growth`** is the annual growth rate from `population_initial` to `population_now`. `pop_growth` is protected and none of the three is missing. Given `population_initial` and `pop_growth`, `population_now` is the derived level and is dropped.
- **`mortality`, `logem4`, `imputedmort`** are one mortality measure in levels, logs and exp(log). `mortality` (51 missing) and `imputedmort` (35) are dropped, and `logem4` (31) is kept.
- **Population density = population / `area`:** in 1, 1000 and 1500 CE, density and population have the same number of missing values (17, 7 and 3), so the derived densities are dropped. `area` has none.
- **Continent dummies** sum to one and none is missing, so the base category `europe` (as in `vv_geography`) is dropped.
- **Religion shares** sum to 100 and none is missing, so the base category `pother` (as in `vv_religion`) is dropped.
- **Barro-Lee shares** split the population by education level. `ls_bl` and `lh_bl` are protected, so `lp_bl` (as in `vv_educ`) is dropped.
- **`polity2`** is `polity` recoded. Both have 16 missing values, but `polity2` corrects the invalid `-88` codes for PAK and SDN. Therefore, `polity` is dropped and `polity2` is retained for the analysis.
- **`distcr`** (distance to coast or river) is a function of `distc` and `distr`. All three have 9 missing, so the derived `distcr` is dropped.

## 5. Timing

The remaining candidates below are measured after 1970 according to their labels. Their values may respond to growth over the period itself (bad controls). They are kept for now, and the group decides:

- `investment_rate` (average 1970-2020) and `pop_growth` (1970-2020), both protected
- `capital_growth_pct_gdp_now` (2020)
- `currentinst` (ease of doing business, 2012)
- `imr95` and `leb95` (infant mortality and life expectancy, 1995)
- `yellow` (yellow fever vector present "today")

Many labels state no year (for example the Barro-Lee shares and the democracy measures), so their timing is unknown.

## 6. Correlation pruning

Two controls with an absolute correlation of 0.95 or more (rounded half up to two decimals) carry almost the same information. Pairs are taken from the highest correlation down. In each pair, the variable with fewer missing values is kept and protected variables are never dropped. Ties are decided by economic interpretation, and pairs that contain an already dropped variable are skipped. `lgdp_initial` is included so that no control can be a near-copy of the regressor. A tiny constant is added before rounding so that exact halves round up despite floating-point representation. Numbers in brackets are missing values in the trimmed sample.

In [9]:
# 1. remaining candidates, including the regressor
vv_cand = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth']]

# 2. absolute correlations above the diagonal, highest first
C = dat_trim[vv_cand].corr().abs()
upper = np.triu(np.ones(C.shape, dtype=bool), k=1)
pairs = C.where(upper).stack().sort_values(ascending=False)

# 3. round half up to two decimals and keep pairs of 0.95 or more
pairs = (pairs + 1e-9).round(2)
pairs = pairs[pairs >= 0.95]

# 4. drop the variable with more missing values; ties are decided by hand
vv_tie = {('demCGV', 'demBMR'): 'demCGV', ('dem', 'demBMR'): 'demBMR'}
rows = []
for (a, b), r in pairs.items():
    if a in vv_dropped or b in vv_dropped or (a in protected and b in protected):
        drop = ''
    elif a in protected or b in protected:
        drop = b if a in protected else a
    elif n_miss[a] != n_miss[b]:
        drop = a if n_miss[a] > n_miss[b] else b
    else:
        drop = vv_tie[(a, b)]
    if drop:
        vv_dropped[drop] = 'correlation'
    rows.append([f'{a} ({n_miss[a]})', f'{b} ({n_miss[b]})', r, drop])

pd.DataFrame(rows, columns=['a', 'b', 'corr', 'dropped'])

,a,b,corr,dropped
0,pop1000 (7),pop1500 (3),0.99,pop1000
1,pdiv (0),pdivhmi (24),0.99,pdivhmi
2,pop1 (17),pop1500 (3),0.98,pop1
3,ginv (16),capital_growth_pct_gdp_initial (17),0.98,capital_growth_pct_gdp_initial
4,pdiv_aa (5),pdivhmi_aa (25),0.97,pdivhmi_aa
5,pop1500 (3),population_initial (0),0.97,pop1500
6,pop1 (17),pop1000 (7),0.96,
7,pop1000 (7),population_initial (0),0.95,
8,demCGV (12),demBMR (12),0.95,demCGV
9,dem (12),demBMR (12),0.95,demBMR


Drops and ties (missing values in brackets):

- `pop1000` (7) and `pop1` (17) are dropped against `pop1500` (3): population levels in different centuries are near-copies of each other. `pop1500` (3) is then dropped against `population_initial` (0), because in levels both are dominated by the same large countries.
- `pdivhmi` (24) and `pdivhmi_aa` (25) are dropped against `pdiv` (0) and `pdiv_aa` (5). They are alternative versions of predicted genetic diversity.
- `capital_growth_pct_gdp_initial` (17) is dropped against `ginv` (16): both measure investment as a share of GDP. With four Solow variables, `capital_growth_pct_gdp_initial` is no longer protected, so the variable with fewer missing values is kept.
- `abslat` (1) is dropped against `uvdamage` (0). UV exposure is largely determined by distance from the equator.
- **Tie, `demCGV` and `demBMR` (12 each).** Both are 0/1 codings of democracy, but `demCGV` also requires that power has changed hands, so a stable democracy where one party keeps winning is coded as non-democratic. `demBMR` is easier to interpret and is kept.
- **Tie, `dem` and `demBMR` (12 each).** `dem` is the source's own democracy measure (ANRR) and `demBMR` an alternative coding of the same concept, so `dem` is kept.
- The other three pairs involve a variable that was already dropped and need no action.

## 7. Missingness

The lasso needs every control observed for every country. Controls missing for more than `MAX_MISSING_SHARE` of the trimmed sample are dropped (protected controls never are), and then only complete cases are kept.

In [10]:
# 1. share of missing values per remaining control in the trimmed sample
MAX_MISSING_SHARE = 0.20
vv_x = [v for v in dat_trim.columns if v not in vv_dropped and v not in ['code', 'gdp_growth', 'lgdp_initial']]
miss = dat_trim[vv_x].isnull().mean()

# 2. drop controls above the threshold, never protected ones
for v in vv_x:
    if miss[v] > MAX_MISSING_SHARE and v not in protected:
        vv_dropped[v] = 'missingness'
print(f"Dropped (missing share above {MAX_MISSING_SHARE}): {[v for v in vv_x if v in vv_dropped]}")

# 3. table of missing shares
pd.DataFrame({'missing share': miss,
              'protected': [v in protected for v in vv_x],
              'dropped': [v in vv_dropped for v in vv_x]}).sort_values('missing share', ascending=False).round(3)

Dropped (missing share above 0.2): ['logem4', 'excolony', 'lt100km', 'democ1', 'democ00a', 'cons00a', 'imr95', 'leb95', 'capital_growth_pct_gdp_now']


,missing share,protected,dropped
leb95,0.510,False,True
imr95,0.510,False,True
lt100km,0.500,False,True
excolony,0.461,False,True
democ1,0.343,False,True
democ00a,0.343,False,True
cons00a,0.314,False,True
logem4,0.304,False,True
capital_growth_pct_gdp_now,0.245,False,True
marketref,0.176,False,False


In [11]:
# 1. remaining controls
vv_x = [v for v in vv_x if v not in vv_dropped]

# 2. keep complete cases on outcome, regressor and controls
J = dat_trim[['gdp_growth', 'lgdp_initial'] + vv_x].notnull().all(axis=1)
dat_pre = dat_trim.loc[J, ['code', 'gdp_growth', 'lgdp_initial'] + vv_x]
print(f'Countries in the trimmed sample: {dat_trim.shape[0]}, complete cases: {dat_pre.shape[0]}')

Countries in the trimmed sample: 102, complete cases: 67


In [12]:
# 1. countries lost and the variables they miss
L = dat_trim.loc[~J, ['code'] + vv_x]
pd.DataFrame({'code': L['code'],
              'n missing': L[vv_x].isnull().sum(axis=1),
              'missing': L[vv_x].isnull().apply(lambda r: ', '.join(r.index[r]), axis=1)}).sort_values(['n missing', 'code'])

,code,n missing,missing
10,BDI,1,marketref
26,BWA,1,marketref
34,COG,1,marketref
42,DEU,1,polity2
99,LSO,1,marketref
111,MMR,1,currentinst
116,MWI,1,investment_rate
129,PAN,1,ginv
149,SLE,1,ginv
13,BFA,2,"ls_bl, lh_bl"


Nine controls are missing for more than `MAX_MISSING_SHARE` of the trimmed sample and are dropped; no protected control is. Of the 102 trimmed countries, 67 observe every remaining variable.

- **Countries lost to a single variable.** Nine countries are lost because of one variable: BDI, BWA, COG and LSO miss only `marketref`; DEU misses only `polity2`; MMR misses only `currentinst`; MWI misses only `investment_rate`; PAN and SLE miss only `ginv`.
- **Small states and territories.** KIR, SYC, VCT and PRI miss most of the geographic and institutional variables.
- **Change in the sample.** The final sample has 67 countries as with five Solow variables, but one country is swapped: ZMB, which missed only `capital_growth_pct_gdp_initial`, is now kept, and PAN, which misses `ginv`, is now lost.

## 8. Kept versus dropped

Table 1 gives the status (kept or dropped) and the label (from `lbl_all`) of every variable and is written to `output/variable_status.tex`. Table 2 compares the countries in the final sample (kept) with the other countries in the trimmed sample (dropped) for the outcome, the regressor and the Solow variables: the mean of each, the number of countries that observe it, and the p-value of a two-sided Welch t-test of equal means in the two groups. The Welch test allows the variances to differ, since the two groups differ in size and spread. The table is written to `output/kept_vs_dropped.tex` with a caption. Countries without `gdp_growth` or `lgdp_initial` can never enter the estimation sample, so they are left out, and the two groups together make up the trimmed sample.

In [13]:
# 1. table 1: status and label of every variable
tab1 = pd.DataFrame({'status': ['dropped' if v in vv_dropped else 'kept' for v in dat.columns],
                     'label': [lbl_all.get(v, '') for v in dat.columns]}, index=dat.columns)

# 2. write table 1 to LaTeX
Path('output').mkdir(exist_ok=True)
Path('output/variable_status.tex').write_text(tabulate(tab1, headers='keys', tablefmt='latex_booktabs'))
tab1

,status,label
code,kept,World Bank country code
marketref,kept,Index of market reforms (1960)
dem,kept,Democracy measure by ANRR
demCGV,dropped,Democracy measure by CGV
demBMR,dropped,Democracy measure by BMR
...,...,...
gdp_growth,kept,"Annual growth in GDP per capita, 1970-2020"
pop_growth,kept,"Annual growth in population, 1970-2020"
lgdp_initial,kept,GDP per capita in 1970 (log)
lpop_initial,dropped,Population in 1970 (log)


In [14]:
# 1. countries in the final sample (kept) and the other countries in the trimmed sample (dropped)
K = dat['code'].isin(dat_pre['code'])
D = dat['code'].isin(dat_trim['code']) & ~K

# 2. table 2: means, counts and Welch t-test of equal means for the outcome, the regressor and the Solow variables
vv = ['gdp_growth', 'lgdp_initial'] + list(vv_solow.values())
tab = pd.DataFrame({'Mean in kept countries': dat.loc[K, vv].mean(),
                    'Mean in dropped countries': dat.loc[D, vv].mean(),
                    'Country kept': dat.loc[K, vv].count(),
                    'Country dropped': dat.loc[D, vv].count(),
                    'p-value': [stats.ttest_ind(dat.loc[K, v], dat.loc[D, v], equal_var=False, nan_policy='omit').pvalue for v in vv]}).round(3)

# 3. write table 2 to LaTeX with a caption
caption = 'Kept versus dropped countries'
note = (r'Mean of each variable and number of countries that observe it, for countries in the final sample (kept) and the other countries in the trimmed sample, which observe both $g_i$ and $y_{i0}$ (dropped). '
        r'The p-value is from a two-sided Welch t-test of equal means in the two groups (unequal variances).')
lines = [r'\begin{table}[!htbp]', r'\centering',
         r'\caption{' + caption + '}', r'\label{tab:kept_vs_dropped}',
         r'\begin{threeparttable}', tabulate(tab, headers='keys', tablefmt='latex_booktabs'),
         r'\begin{tablenotes}[flushleft]', r'\footnotesize',
         r'\item \textit{Note:} ' + note,
         r'\item \textit{Source:} Own calculations based on provided growth data.',
         r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
Path('output').mkdir(exist_ok=True)
Path('output/kept_vs_dropped.tex').write_text('\n'.join(lines))
tab.style.set_caption(caption).format(precision=3)

,Mean in kept countries,Mean in dropped countries,Country kept,Country dropped,p-value
gdp_growth,0.016,0.016,67,35,0.950
lgdp_initial,7.967,7.490,67,35,0.093
pop_growth,0.017,0.018,67,35,0.842
investment_rate,22.588,23.748,67,32,0.395
lh_bl,3.126,1.511,67,21,0.009
ls_bl,12.750,8.007,67,21,0.045


Kept and dropped countries grew at the same rate (mean `gdp_growth` 0.016 in both groups, p = 0.95), so the restrictions do not select on the outcome. Dropped countries started somewhat poorer (mean `lgdp_initial` 7.967 against 7.49, p = 0.093), a difference that is significant at the 10% level but not at the 5% level.

- **Similar.** Population growth (0.017 against 0.018, p = 0.842) and the average investment rate (`investment_rate` 22.588 against 23.748, p = 0.395) do not differ significantly.
- **Different.** Kept countries have a higher secondary education share (`ls_bl` 12.75 against 8.007, p = 0.045) and a higher tertiary share (`lh_bl` 3.126 against 1.511, p = 0.009). With a Bonferroni correction for the six tests (0.05/6 = 0.008), neither gap is significant, although the tertiary gap is just above the threshold.
- **Coverage.** The dropped group is small for the education shares, which only 21 of the 35 dropped countries observe, so those means and tests rest on few countries.

Countries are dropped for missing controls, and the dropped countries are less educated and somewhat poorer, so the estimate of $\beta$ describes a final sample that is more educated than the trimmed sample as a whole. The comparison covers observed characteristics only and says nothing about differences in unobservables.

## 9. Summary

Final counts, the final control names in `vv_prescreened`, and the prescreened data `dat_pre` saved to `output/growth_prescreened.csv` for the estimation step.

In [15]:
# 1. final counts
vv_prescreened = [v for v in dat_pre.columns if v not in ['code', 'gdp_growth', 'lgdp_initial']]
print(f'Countries (rows): {dat_pre.shape[0]}')
print(f'Explanatory variables (columns minus gdp_growth and code): {dat_pre.shape[1] - 2}')
print(f'Controls excluding lgdp_initial: {len(vv_prescreened)}')

# 2. save the prescreened data
dat_pre.to_csv('output/growth_postselection.csv', index=False)

Countries (rows): 67
Explanatory variables (columns minus gdp_growth and code): 49
Controls excluding lgdp_initial: 48


# Simple OLS

In [16]:
# 1. avoiding missings 
I = dat[['gdp_growth', 'lgdp_initial']].notnull().all(axis=1)

# 2. extract dataset 
y = dat.loc[I, 'gdp_growth'].values.reshape((-1,1)) * 100.0
X = dat.loc[I, ['constant','lgdp_initial']].values

# 3. run OLS
betahat = np.linalg.inv(X.T @ X) @ X.T @ y
print(betahat)

[[ 2.24792211]
 [-0.08713416]]


# Adding more controls

In [17]:
vs = vv_all['geography'] + vv_all['religion']
xs = ['lgdp_initial', 'pop_growth', 'investment_rate'] + vs

# avoiding missings
all_vars = ['gdp_growth'] + xs
I = dat[all_vars].notnull().all(1)

# extract data
X = dat.loc[I, xs].values
y = dat.loc[I,'gdp_growth'].values.reshape((-1,1)) * 100. #easier to read output when growth is in 100%

# add const. (unless this breaks the rank condition)
oo = np.ones((I.sum(),1))
X = np.hstack([X, oo])
xs.append('constant') # we put it in as the last element

# check the rank condition
K = X.shape[1]
assert np.linalg.matrix_rank(X) == X.shape[1], f'X does not have full rank'

# compute the OLS estimator
betas = np.linalg.inv(X.T @ X) @ X.T @ y

/var/folders/_f/ks6p8wfd7k3fr788qzkk3tfm0000gn/T/ipykernel_74006/2288867945.py:6: Pandas4Warning: Starting with pandas version 4.0 all arguments of all will be keyword-only.
  I = dat[all_vars].notnull().all(1)


In [18]:
# format nicely
print(f'Mean y = {y.mean(): 5.2f}% growth per year')
pd.DataFrame({'β': betas[:,0]}, index=xs).round(3)

Mean y =  1.52% growth per year


,β
lgdp_initial,-1.118
pop_growth,6.830
investment_rate,0.068
tropicar,-0.411
distr,-0.000
distcr,0.001
distc,-0.003
suitavg,-1.063
temp,-0.204
suitgini,-0.771


## Estimation

### Post double Lasso

In [19]:
from sklearn.linear_model import Lasso

# 0. estimation data and variables from the selection of controls
dat_est = dat_pre                                           # complete cases
y_name  = 'gdp_growth'                                      # outcome, Y
d_name  = 'lgdp_initial'                                    # regressor of interest, D
z_names = [v for v in vv_prescreened if v != 'constant']    # candidate controls, Z (Lasso adds the intercept)

# 1. extract data
y = dat_est[y_name].values * 100.0 # growth in percent per year, as in the OLS above
d = dat_est[d_name].values
Z = dat_est[z_names].values
X = np.column_stack((d, Z))
N = X.shape[0]

# 2. standardize (ddof=1 degrees of freedom correction)
def standardize(X):
    X_stan = (X - np.mean(X, axis=0))/np.std(X, axis=0, ddof=1)
    return X_stan

X_stan = standardize(X)
Z_stan = standardize(Z)
d_stan = standardize(d)

# 3. BCCH penalty (robust to heteroskedasticity)
def BCCH(X_tilde, y, c=1.1, alpha=0.05):
    (N, p) = X_tilde.shape

    # pilot penalty
    yXscale = (np.max((X_tilde.T ** 2) @ ((y - np.mean(y)) ** 2) / N)) ** 0.5
    penalty_pilot = c / np.sqrt(N) * stats.norm.ppf(1-alpha/(2*p)) * yXscale # Note: divided by 2 due to Python definition of Lasso

    # pilot residuals
    pred = Lasso(alpha=penalty_pilot, max_iter=10000).fit(X_tilde, y).predict(X_tilde)
    eps = y - pred

    # updated penalty
    epsXscale = (np.max((X_tilde.T ** 2) @ (eps ** 2) / N)) ** 0.5
    penalty_BCCH = c / np.sqrt(N) * stats.norm.ppf(1-alpha/(2*p)) * epsXscale
    return penalty_BCCH

# 4. step 1: Lasso Y on D and Z
penalty_BCCHyx = BCCH(X_stan, y)
fit_BCCHyx = Lasso(penalty_BCCHyx, max_iter=10000).fit(X_stan, y)
coefs_yx = fit_BCCHyx.coef_

# residuals and Y - Z@gamma (epsilon + alpha*d)
resyx = y - fit_BCCHyx.predict(X_stan)
resyxz = resyx + d_stan*coefs_yx[0]

# 5. step 2: Lasso D on Z
penalty_BCCHdz = BCCH(Z_stan, d)
fit_BCCHdz = Lasso(penalty_BCCHdz, max_iter=10000).fit(Z_stan, d)
coefs_dz = fit_BCCHdz.coef_

# residuals
resdz = d - fit_BCCHdz.predict(Z_stan)

# 6. step 3: estimate alpha
alpha_PDL = (resdz@resyxz)/(resdz@d)

# 7. variance (heteroskedasticity robust) and standard error
sigma2_PDL = (resdz**2@resyx**2/N)/((resdz@resdz/N)**2)
se_PDL = np.sqrt(sigma2_PDL/N)

# 8. 95% confidence interval
q = stats.norm.ppf(1-0.025)
CI_low_PDL  = alpha_PDL - q*se_PDL
CI_high_PDL = alpha_PDL + q*se_PDL

# 9. selected controls in each step
vv_sel_yx = [z for z, b in zip(z_names, coefs_yx[1:]) if b != 0]
vv_sel_dz = [z for z, b in zip(z_names, coefs_dz) if b != 0]
print(f'N = {N}, candidate controls = {len(z_names)}')
print(f'lambda_BCCH (Y on D, Z): {penalty_BCCHyx:.4f}, selected controls: {vv_sel_yx}')
print(f'lambda_BCCH (D on Z):    {penalty_BCCHdz:.4f}, selected controls: {vv_sel_dz}')

# 10. LaTeX table (the functions are reused for the PPOL table below)
def fmt(x, k=3):
    return f'{x:.{k}f}'.replace('-', '$-$')

def stars(b, se):
    p = 2*(1 - stats.norm.cdf(abs(b/se))) # two-sided test against zero
    return '$^{***}$' if p < 0.01 else '$^{**}$' if p < 0.05 else '$^{*}$' if p < 0.1 else ''

def latex_table(caption, label, b_label, b, se, ci_low, ci_high, rows_sel, N, p, note):
    lines = [r'\begin{table}[!htbp]', r'\centering',
             r'\caption{' + caption + '}', r'\label{' + label + '}',
             r'\begin{threeparttable}', r'\begin{tabular}{lc}', r'\toprule',
             r' & Annual growth in GDP p.c. (\%) \\', r'\midrule',
             b_label + ' & ' + fmt(b) + stars(b, se) + r' \\',
             r' & (' + fmt(se) + r') \\', r'\addlinespace',
             r'95\% confidence interval & [' + fmt(ci_low) + ', ' + fmt(ci_high) + r'] \\', r'\midrule']
    lines += [k + ' & ' + v + r' \\' for k, v in rows_sel]
    lines += [r'\midrule',
              r'Observations (countries) & ' + str(N) + r' \\',
              r'Candidate controls & ' + str(p) + r' \\',
              r'\bottomrule', r'\end{tabular}',
              r'\begin{tablenotes}[flushleft]', r'\footnotesize',
              r'\item \textit{Note:} ' + note,
              r'\item \textit{Source:} Own calculations based on provided growth data.',
              r'\end{tablenotes}', r'\end{threeparttable}', r'\end{table}']
    return '\n'.join(lines)

tex_PDL = latex_table(
    caption=r'Post double Lasso estimate of $\beta$',
    label='tab:pdl_estimates',
    b_label=r'Log GDP p.c. in 1970, $\check{\beta}$',
    b=alpha_PDL, se=se_PDL, ci_low=CI_low_PDL, ci_high=CI_high_PDL,
    rows_sel=[(r'Penalty $\hat{\lambda}$, growth on initial GDP and controls', fmt(penalty_BCCHyx)),
              (r'Penalty $\hat{\lambda}$, initial GDP on controls', fmt(penalty_BCCHdz)),
              ('Selected controls, growth equation', str(len(vv_sel_yx))),
              ('Selected controls, initial GDP equation', str(len(vv_sel_dz)))],
    N=N, p=len(z_names),
    note=r'Post double Lasso estimate of $\beta$ in $g_i = \beta y_{i0} + \mathbf{z}_i\boldsymbol{\gamma} + u_i$. '
         r'BCCH penalties ($c=1.1$, $\alpha=0.05$) on standardized regressors. '
         r'Heteroskedasticity-robust standard errors in parentheses. '
         r'$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$, for tests against zero.')

Path('output').mkdir(exist_ok=True)
Path('output/pdl_estimates.tex').write_text(tex_PDL)

# 11. results
pd.DataFrame({'Estimate of alpha': alpha_PDL, 'Standard error': se_PDL,
              'Low bound of CI': CI_low_PDL, 'High bound of CI': CI_high_PDL}, index=['PDL']).round(4)

N = 67, candidate controls = 47
lambda_BCCH (Y on D, Z): 2.0931, selected controls: []
lambda_BCCH (D on Z):    1.0113, selected controls: []


,Estimate of alpha,Standard error,Low bound of CI,High bound of CI
PDL,-0.1538,0.1491,-0.446,0.1385


#### Issues to be aware of

- **`constant` in the control list.** `vv_prescreened` (and `vv_x`) contains `constant`, the column of ones added after loading the data. It has zero variance, so standardizing it divides by zero and gives NaN, and the Lasso adds its own intercept. It is removed in the estimation cell only (`z_names`), so the controls are 47, not 48. It is still in `output/growth_postselection.csv` and in the count printed in section 9.
- **`oceania` covers one country.** In the 67-country sample only AUS has `oceania` = 1. After standardizing, it is close to a dummy for that single country (standardized value 8.06).
- **No controls selected with BCCH.** With the BCCH penalty, neither Lasso step selects any control. Post double Lasso then reduces to OLS of growth on `lgdp_initial` alone in the 67-country sample (−0.154, the same as the simple OLS on this sample). With N = 67 and p = 48, the penalty is high relative to the signal in the data.
- **Variables in levels with extreme values raise the BCCH penalty.** The BCCH penalty uses the largest value of $\sqrt{\frac{1}{N}\sum_i x_{ij}^2 \hat{\varepsilon}_i^2}$ over the controls $j$. A control with one extreme country, and a large residual for that country, sets the penalty for all controls. `population_initial` (in levels) is 6.44 standard deviations above the mean for CHN, which also grew fast. This makes the Y-equation pilot scale 4.75, against a standard deviation of growth of 1.43. Other skewed level variables are `area`, `area_ar` and `oilres` (SAU, 7.98). As a diagnostic only, dropping `population_initial` lowers the penalties, and then the D-equation selects `uvdamage`, while the Y-equation still selects nothing. The group may consider entering population in logs (`lpop_initial`, which section 4 drops) or logging other skewed controls.
- **Validity of the standard error.** The standard error is robust to heteroskedasticity and uses the step 1 residuals. It is valid if the controls selected in the two steps approximate both equations well (approximate sparsity). With no controls selected, the result relies on all omitted controls being unimportant for either growth or initial income.
- **Timing of controls.** `investment_rate` and `pop_growth` (1970–2020, protected), `currentinst` (2012) and `yellow` (today) are measured after 1970 and may respond to growth over the period (bad controls). See section 5.
- **Run order.** The estimation cell needs `dat_pre` and `vv_prescreened` from sections 1–9, and it overwrites `y`, `X` and `N` from the OLS cells above.
- **Notation.** The code calls the estimate `alpha_PDL` (as in the course exercises). In the assignment's notation it is $\beta$, the convergence coefficient.

### Post partialling out Lasso

In [20]:
# Note: uses y, d, Z_stan, N, z_names, BCCH(), fmt() and latex_table() from the post double Lasso cell above

# 1. step 1: Lasso Y on Z
penalty_BCCHyz = BCCH(Z_stan, y)
fit_BCCHyz = Lasso(penalty_BCCHyz, max_iter=10000).fit(Z_stan, y)
coefs_yz = fit_BCCHyz.coef_

# residuals
resyz = y - fit_BCCHyz.predict(Z_stan)

# 2. step 2: Lasso D on Z
penalty_BCCHdz = BCCH(Z_stan, d)
fit_BCCHdz = Lasso(penalty_BCCHdz, max_iter=10000).fit(Z_stan, d)
coefs_dz = fit_BCCHdz.coef_

# residuals
resdz = d - fit_BCCHdz.predict(Z_stan)

# 3. step 3: estimate alpha (OLS of resyz on resdz)
alpha_PPOL = (resdz@resyz)/(resdz@resdz)

# 4. variance and standard error
sigma2_PPOL = (resyz**2@resdz**2/N)/((resdz@resdz/N)**2)
se_PPOL = np.sqrt(sigma2_PPOL/N)

# 5. 95% confidence interval
q = stats.norm.ppf(1-0.025)
CI_low_PPOL  = alpha_PPOL - q*se_PPOL
CI_high_PPOL = alpha_PPOL + q*se_PPOL

# 6. selected controls in each step
vv_sel_yz = [z for z, b in zip(z_names, coefs_yz) if b != 0]
vv_sel_dz = [z for z, b in zip(z_names, coefs_dz) if b != 0]
print(f'N = {N}, candidate controls = {len(z_names)}')
print(f'lambda_BCCH (Y on Z): {penalty_BCCHyz:.4f}, selected controls: {vv_sel_yz}')
print(f'lambda_BCCH (D on Z): {penalty_BCCHdz:.4f}, selected controls: {vv_sel_dz}')

# 7. LaTeX table
tex_PPOL = latex_table(
    caption=r'Post partialling out Lasso estimate of $\beta$',
    label='tab:ppol_estimates',
    b_label=r'Log GDP p.c. in 1970, $\breve{\beta}$',
    b=alpha_PPOL, se=se_PPOL, ci_low=CI_low_PPOL, ci_high=CI_high_PPOL,
    rows_sel=[(r'Penalty $\hat{\lambda}$, growth on controls', fmt(penalty_BCCHyz)),
              (r'Penalty $\hat{\lambda}$, initial GDP on controls', fmt(penalty_BCCHdz)),
              ('Selected controls, growth equation', str(len(vv_sel_yz))),
              ('Selected controls, initial GDP equation', str(len(vv_sel_dz)))],
    N=N, p=len(z_names),
    note=r'Post partialling out Lasso estimate of $\beta$ in $g_i = \beta y_{i0} + \mathbf{z}_i\boldsymbol{\gamma} + u_i$. '
         r'BCCH penalties ($c=1.1$, $\alpha=0.05$) on standardized regressors. '
         r'Heteroskedasticity-robust standard errors in parentheses. '
         r'$^{*}p<0.1$, $^{**}p<0.05$, $^{***}p<0.01$, for tests against zero.')

Path('output').mkdir(exist_ok=True)
Path('output/ppol_estimates.tex').write_text(tex_PPOL)

# 8. results: PDL and PPOL
results = {'PDL' : [alpha_PDL,  se_PDL,  CI_low_PDL,  CI_high_PDL],
           'PPOL': [alpha_PPOL, se_PPOL, CI_low_PPOL, CI_high_PPOL]}
pd.DataFrame.from_dict(results, orient='index',
                       columns=['Estimate of alpha', 'Standard error', 'Low bound of CI', 'High bound of CI']).round(4)

N = 67, candidate controls = 47
lambda_BCCH (Y on Z): 2.0893, selected controls: []
lambda_BCCH (D on Z): 1.0113, selected controls: []


,Estimate of alpha,Standard error,Low bound of CI,High bound of CI
PDL,-0.1538,0.1491,-0.446,0.1385
PPOL,-0.1538,0.1491,-0.446,0.1385


#### Issues to be aware of

- **Run order.** The cell reuses `y`, `d`, `Z_stan`, `N`, `z_names` and `BCCH()` from the post double Lasso cell, so that cell must run first. The step 2 Lasso (D on Z) is the same in both methods, so the cell overwrites `penalty_BCCHdz`, `fit_BCCHdz`, `coefs_dz` and `resdz` with identical values.
- **PPOL and PDL are identical here.** With the BCCH penalty, neither Lasso step selects any control (the Y on Z penalty is 2.089). Both residuals are then deviations from the mean, and both methods reduce to OLS of growth on `lgdp_initial` alone in the 67-country sample (−0.154). The agreement follows from the empty selection and is not a robustness check of PDL.
- **Variance formula from the course exercise.** As in the exercise, the PPOL variance uses the step 1 residuals $\hat{\zeta}_i$, i.e. $\breve{\sigma}^2 = \frac{N^{-1}\sum_i \hat{\zeta}_i^2 \hat{\nu}_i^2}{(N^{-1}\sum_i \hat{\nu}_i^2)^2}$. The orthogonal moment condition $E[(D - Z'\psi_0)([Y - Z'\delta_0] - \alpha_0[D - Z'\psi_0])] = 0$ has the error $\hat{\zeta}_i - \breve{\alpha}\hat{\nu}_i$ in place of $\hat{\zeta}_i$. The two coincide only if $\alpha_0 = 0$. With $\hat{\zeta}_i - \breve{\alpha}\hat{\nu}_i$, the standard error here is 0.1436 instead of 0.1491. The difference is small because the estimate is close to zero, but the group should decide which formula to report and state it.
- **The same applies to the PDL standard error.** PDL uses the residuals from the step 1 Lasso of Y on D and Z, and D is penalized in that step. Here its coefficient is set to zero, so those residuals are also deviations from the mean of growth, and the PDL standard error (0.1491) has the same issue.